# Spine MRI Decision-Support API — Colab run + test

**Use this notebook (`colab_api_run.ipynb`) to *run and test* the pipeline + REST API.**

- **To get real vision weights first** (once per project), use `colab_training.ipynb` and save
  the checkpoint as `checkpoints/best_model.pth`. Without it, this notebook still runs the
  whole flow in an honest, clearly-labelled **MOCK vision** demo mode.
- Cells below: clone repo → install deps → set Gemini key → smoke test → test with the
  real AI (Gemini report) → start the API → hit it → get a public URL for your frontend.

In [ ]:
import os

# 1) Clone the repo. If you already uploaded the project, skip this and just
#    %cd to it.
REPO_URL = "https://github.com/codermisba/spinelit-ai.git"

if not os.path.exists('/content/spine-foundation'):
    !git clone "$REPO_URL" /content/spine-foundation
os.chdir('/content/spine-foundation')
print('Working dir:', os.getcwd())
!ls

In [ ]:
# 2) Dependencies (torch/torchvision are preinstalled on Colab).
!pip install -q timm pillow numpy pandas pydantic pyyaml scikit-learn tqdm \
    fastapi uvicorn python-multipart nest-asyncio requests

In [ ]:
# 3) Gemini API key — add it in the Colab SECRETS panel (key name: GEMINI_API_KEY).
#    This powers real AI report generation; without it the pipeline still returns
#    a safe conservative report.
from google.colab import userdata
try:
    key = userdata.get('GEMINI_API_KEY')
    with open('.env', 'w') as f:
        f.write(f'GEMINI_API_KEY={key}\n')
    print('API key set (len', len(key), ')')
except Exception as e:
    print('No key set:', e, '— reports will use the deterministic fallback.')

In [ ]:
# 4) Vision checkpoint. Optional path from Google Drive, plus status.
import os, shutil
os.makedirs('checkpoints', exist_ok=True)

CKPT = 'checkpoints/best_model.pth'
if not os.path.exists(CKPT):
    drive_src = '/content/drive/MyDrive/spine_foundation/checkpoints/best_model.pth'
    if os.path.exists(drive_src):
        shutil.copy(drive_src, CKPT)
        print('Copied checkpoint from Drive.')
    else:
        print('No checkpoint yet → running in MOCK vision demo mode.')
        print('Train real weights with colab_training.ipynb and save', CKPT)
else:
    print('Checkpoint found:', CKPT)

In [ ]:
# 5) Smoke test — deterministic decision-support path (no LLM, no checkpoint).
from decision_support import run_example, run_decision_support

r = run_example(use_llm=False)
print('overall_status      :', r.case.overall_status.value)
print('report_validated    :', r.report_validated)
print('traceability entries:', len(r.report.traceability))
print('sample statuses     :', sorted({f.evidence_status.value for f in r.case.findings}))

In [ ]:
# 6) TEST WITH A GOOD AI — generate the report with Gemini on the same case.
#    If Gemini is transiently busy (HTTP 503) the pipeline retries then safely
#    falls back to the validated conservative report (llm_used=False).
r_llm = run_example(use_llm=True)
print('llm_used            :', r_llm.llm_used)
print('report_validated    :', r_llm.report_validated)
print()
print(r_llm.report.text[:1600])

In [ ]:
# 7) Start the REST API in the background (for the frontend).
import nest_asyncio, threading, time, uvicorn, requests
nest_asyncio.apply()

STARTED = False
def _serve():
    uvicorn.run('api_server:app', host='0.0.0.0', port=8000, log_level='warning')

if not any(t.name == 'spine_api' for t in threading.enumerate()):
    t = threading.Thread(target=_serve, name='spine_api', daemon=True)
    t.start()

for _ in range(20):
    try:
        h = requests.get('http://127.0.0.1:8000/api/health', timeout=3).json()
        print('API up:', h)
        STARTED = True
        break
    except Exception:
        time.sleep(2)
if not STARTED:
    raise RuntimeError('API failed to start')

In [ ]:
# 8) Test the API end-to-end: image -> evidence -> report -> annotated image.
import base64, io
from PIL import Image

BASE = 'http://127.0.0.1:8000'

# Use your own spine image: upload it, or drop it below instead of this placeholder.
sample = Image.new('RGB', (256, 256), (12, 12, 38))
buf = io.BytesIO(); sample.save(buf, format='PNG'); buf.seek(0)

resp = requests.post(f"{BASE}/api/analyze",
    files={'file': ('scan.png', buf, 'image/png')},
    data={
        'use_llm': 'false',            # set 'true' to use Gemini for the report
        'mock_vision': 'true',         # auto-true until a checkpoint exists
        'clinical': '{"age": 52, "pain_score": 5, "pain_duration_years": 4, '
                    '"symptoms": {"right_leg_numbness": true}}',
    })
resp.raise_for_status()
data = resp.json()

print('overall_status :', data['case']['overall_status'])
print('report_validated:', data['report_validated'], '| mock_vision:', data['mock_vision'])
print('impression     :', data['report']['impression'][:1])

png = base64.b64decode(data['annotated_image_data_url'].split(',')[1])
open('outputs/api_sample_annotated.png', 'wb').write(png)
Image.open('outputs/api_sample_annotated.png')

In [ ]:
# 9) Public URL for your frontend (optional).
#    Free option A: ngrok  |  Free option B: cloudflared   (pick one)
A = False
if A:
    from google.colab import userdata
    from pyngrok import ngrok
    token = userdata.get('NGROK_TOKEN', '')
    if token:
        ngrok.set_auth_token(token)
    t = ngrok.connect(8000)
    print('Public URL (frontend):', t.public_url)
else:
    print('Run in a cell:  !cloudflared tunnel --url http://localhost:8000')
    print('then send the printed https URL to your frontend.')

## Frontend contract

### `GET {BASE}/api/health`
`status`, `vision.{loaded,checkpoint,error}`, `gemini_configured`.

### `POST {BASE}/api/analyze`  (multipart/form-data)
| field | type | meaning |
|---|---|---|
| `file` | file | spine image (PNG/JPG) |
| `use_llm` | str | `"true"` uses Gemini for the report |
| `provider` | str | usually empty (=default) |
| `case_id` | str | optional |
| `clinical` | json str | `{age, sex, pain_score, pain_duration_years, symptoms:{...}, image_quality:{...}}` |
| `spondy_predictions` | json str | optional slip predictions list |
| `mock_vision` | str | `"true"/"false"/""` (auto) |

Response = a `DecisionSupportResult` JSON plus `evidence`, `model_status`,
`mock_vision` and `annotated_image_data_url` (PNG data-URL with the thin-line
anatomy overlay — ready to drop straight into an `<img src>`).

Typical `fetch` from a browser app:

```js
const fd = new FormData();
fd.append('file', fileInput.files[0]);
fd.append('use_llm', 'true');
fd.append('clinical', JSON.stringify({
  age: 52, pain_score: 5,
  symptoms: { right_leg_numbness: true }
}));
const r = await fetch('https://<public-url>/api/analyze', {
  method: 'POST', body: fd
});
const out = await r.json();
imgEl.src = out.annotated_image_data_url;      // thin-line overlay
statusEl.textContent = out.report.overall_status;
```